# Step 1b — Refresh actuals up to ORIGIN (for the new-origin forecast and the origin sweep)

Same source table and same cleaning as Step 1 (`chunking_code.ipynb`). Differences:

- **Nothing from the original run is overwritten.** The vocabulary (`category_vocab.json`) and `column_roles.json` are only *read*: the embedding tables in the trained model are indexed by that vocabulary, so it must not change.
- Only the window from `WINDOW_START` to `ORIGIN` is written. `WINDOW_START` is early enough that **any origin from `EARLIEST_ORIGIN` to `ORIGIN` still has 365 days of history before it**, so the forecast notebook can be run from several origins (the origin sweep) without re-pulling data.
- The calendar is rebuilt **through Dec 31, 2026** with the same recipe and checked against the old calendar on the overlapping dates.

Outputs: `local_train_data_refresh/chunk_XXXX.parquet` and `calendar_extended_to_dec31.parquet`.

In [ ]:
import os, sys, json, gc, shutil
import numpy as np
import pandas as pd

sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration

from snowflake.snowpark.session import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StringType

session = Session.builder.configs(Snowflake_configuration.snowflake_conn_prop).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

### Configuration

In [ ]:
BASE = r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in October\_1_Data preparation"

# READ-ONLY inputs from the original run
ROLES_PATH     = os.path.join(BASE, "column_roles.json")
OLD_CHUNK_DIR  = os.path.join(BASE, "local_train_data")
OLD_CALENDAR   = os.path.join(BASE, "shared_calendar.parquet")

# NEW outputs (separate names, nothing overwritten)
NEW_CHUNK_DIR  = os.path.join(BASE, "local_train_data_refresh")
NEW_CALENDAR   = os.path.join(BASE, "calendar_extended_to_dec31.parquet")

ORIGIN           = pd.Timestamp("2026-11-03")      # last COMPLETE actual day (check the daily totals in 2b before trusting the last 2-3 days)
EARLIEST_ORIGIN  = pd.Timestamp("2026-10-03")      # earliest origin you may want to forecast from (origin sweep)
FORECAST_END_NEW = pd.Timestamp("2026-12-31")
CHUNK_SIZE       = 100                              # dealers per parquet file (same as Step 1)

with open(ROLES_PATH) as f:
    roles = json.load(f)
with open(os.path.join(BASE, roles["vocab_path"])) as f:
    vocab = json.load(f)

time_col, group_col, dealer_col, target_col = roles["time_col"], roles["group_col"], roles["dealer_col"], roles["target_col"]
FREQ                = roles["freq"]
static_covariates   = roles["static_covariates"]
festive_covariates  = roles["festive_covariates"]
calendar_covariates = roles["calendar_covariates"]
future_covariates   = roles["future_covariates"]
TRAIN_START         = pd.Timestamp(roles["train_start"])
VAL_END             = pd.Timestamp(roles["val_end"])
INPUT_CHUNK_LENGTH  = roles["input_chunk_length"]
OUTPUT_CHUNK_LENGTH = roles["output_chunk_length"]

WINDOW_START = EARLIEST_ORIGIN - pd.Timedelta(days=INPUT_CHUNK_LENGTH - 1)   # 365 days of history before the earliest origin
WINDOW_DAYS  = (ORIGIN - WINDOW_START).days + 1
assert EARLIEST_ORIGIN <= ORIGIN
N_AHEAD      = (FORECAST_END_NEW - ORIGIN).days

assert ORIGIN > VAL_END, "ORIGIN must be later than the original history end."
assert N_AHEAD <= OUTPUT_CHUNK_LENGTH, f"{N_AHEAD} days ahead exceeds output_chunk_length={OUTPUT_CHUNK_LENGTH}."
print(f"data written   : {WINDOW_START.date()} -> {ORIGIN.date()} ({WINDOW_DAYS} days per series)")
print(f"usable origins : {EARLIEST_ORIGIN.date()} -> {ORIGIN.date()} (each has {INPUT_CHUNK_LENGTH} days of history)")
print(f"predict        : {(ORIGIN + pd.Timedelta(days=1)).date()} -> {FORECAST_END_NEW.date()} (n = {N_AHEAD})")

### Read the table (same query and cleaning as Step 1)

In [ ]:
df = session.sql("""SELECT *
                 FROM MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES
                 WHERE PARENT_DEALER_CODE_MODEL_FAMILY IN
                 (SELECT PARENT_DEALER_CODE_MODEL_FAMILY
                  FROM MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING)""")

df = df.drop("MODEL_NAME", "MODEL_FAMILY_CODE", "FIRST_SALE_DATE")
df = df.with_column("DEALER_CITY", F.initcap(F.trim(F.col("DEALER_CITY"))))

# strip quote characters Snowflake keeps in some column names, and normalise the series key
for old in df.columns:
    new = old.replace('"', "")
    if new != old:
        df = df.rename(old, new)
df = df.with_column(group_col, F.replace(F.col(group_col), F.lit("<>"), F.lit("_")))

missing_cols = [c for c in [time_col, group_col, target_col] + static_covariates + future_covariates
                if c not in [x.replace('"', "") for x in df.columns]
                and c not in calendar_covariates]
assert not missing_cols, f"table no longer has these columns: {missing_cols}"

window = df.filter((F.col(time_col) >= str(WINDOW_START.date())) & (F.col(time_col) <= str(ORIGIN.date())))
n_series = window.select(F.count_distinct(group_col)).collect()[0][0]
print("series in window:", f"{n_series:,}")

### Checks on the refreshed window
Everything here must pass before the data goes to the forecast notebook. The most likely real-world failure is a **partly loaded last day** (some dealers not yet reporting the last day or two), so look at the daily totals printed in 2b.

In [ ]:
# --- 2a target ---------------------------------------------------------------
r = window.select(
    F.count(F.lit(1)).alias("N_ROWS"),
    F.sum(F.when(F.col(target_col).is_null(), 1).otherwise(0)).alias("N_NULL"),
    F.sum(F.when(F.col(target_col) < 0, 1).otherwise(0)).alias("N_NEG"),
    F.sum(F.when(F.col(target_col) != F.round(F.col(target_col)), 1).otherwise(0)).alias("N_NON_INTEGER"),
).collect()[0].as_dict()
print("2a target:", r)
assert r["N_NULL"] == 0 and r["N_NEG"] == 0, "Target has nulls or negatives inside the window."
assert r["N_NON_INTEGER"] == 0, "Target has non-integer values."

# --- 2b every series has every day from WINDOW_START to ORIGIN, nothing partial ----------
span = (window.group_by(group_col)
              .agg(F.count(F.lit(1)).alias("N"), F.count_distinct(time_col).alias("ND"),
                   F.min(time_col).alias("D0"), F.max(time_col).alias("D1")))
r = span.select(
    F.sum(F.when(F.col("N") != WINDOW_DAYS, 1).otherwise(0)).alias("N_BAD_COUNT"),
    F.sum(F.when(F.col("ND") != WINDOW_DAYS, 1).otherwise(0)).alias("N_BAD_DISTINCT_DAYS"),
    F.sum(F.when(F.col("D0") != F.lit(str(WINDOW_START.date())), 1).otherwise(0)).alias("N_BAD_START"),
    F.sum(F.when(F.col("D1") != F.lit(str(ORIGIN.date())), 1).otherwise(0)).alias("N_BAD_END"),
).collect()[0].as_dict()
print("2b continuity:", r)
assert all((v or 0) == 0 for v in r.values()), "Some series do not have every day from WINDOW_START to ORIGIN."

daily = (window.group_by(time_col)
               .agg(F.count(F.lit(1)).alias("N_SERIES"), F.sum(target_col).alias("TOTAL_SALES"))
               .sort(time_col).to_pandas())
daily[time_col] = pd.to_datetime(daily[time_col])
print(daily.tail(14).to_string(index=False))
last, prev = daily.iloc[-1], daily.iloc[-8:-1]
print(f"\nORIGIN total vs mean of the previous 7 days: {last['TOTAL_SALES'] / prev['TOTAL_SALES'].mean():.2f}x  (a value far below ~1 means a partial load)")

# --- 2c / 2d static columns constant within a series, no nulls ------------------
per_series = window.group_by(group_col).agg(*[F.count_distinct(F.col(c)).alias(f"S{i}") for i, c in enumerate(static_covariates)])
mx = per_series.select(*[F.max(F.col(f"S{i}")).alias(f"S{i}") for i in range(len(static_covariates))]).collect()[0].as_dict()
not_constant = [c for i, c in enumerate(static_covariates) if (mx[f"S{i}"] or 0) > 1]
nulls = window.select(*[F.sum(F.when(F.col(c).is_null(), 1).otherwise(0)).alias(f"S{i}") for i, c in enumerate(static_covariates)]).collect()[0].as_dict()
has_null = [c for i, c in enumerate(static_covariates) if (nulls[f"S{i}"] or 0) > 0]
print("2c static columns varying within a series:", not_constant or "none")
print("2d static columns with nulls             :", has_null or "none")
assert not not_constant and not has_null

# --- 2e categories must exist in the ORIGINAL vocabulary ------------------------
unseen = {}
for c in static_covariates:
    vals = {str(r[0]) for r in window.select(F.col(c)).distinct().collect()}
    new = sorted(vals - set(vocab[c]))
    if new:
        unseen[c] = new
print("2e categories not in the training vocabulary (they would map to __UNKNOWN__):", unseen or "none")

### Write the chunks (dealer-wise, same as Step 1, window only)

In [ ]:
shutil.rmtree(NEW_CHUNK_DIR, ignore_errors=True)
os.makedirs(NEW_CHUNK_DIR)

chunk_cols = [time_col, group_col, target_col] + static_covariates
dealers = sorted(r[0] for r in window.select(dealer_col).distinct().collect())
chunks = [dealers[i:i + CHUNK_SIZE] for i in range(0, len(dealers), CHUNK_SIZE)]
print(f"{len(dealers)} dealers -> {len(chunks)} chunks")

tot = dict(only_old=0, only_new=0, changed=0, compared=0)
for i, chunk in enumerate(chunks):
    part = window.select(*chunk_cols).filter(F.col(dealer_col).isin(chunk)).to_pandas()
    part[time_col] = pd.to_datetime(part[time_col])
    part[target_col] = part[target_col].astype(np.float32)
    part.to_parquet(os.path.join(NEW_CHUNK_DIR, f"chunk_{i:04d}.parquet"), index=False)

    # history that already existed must not have changed (late corrections / restatements)
    old_path = os.path.join(OLD_CHUNK_DIR, f"chunk_{i:04d}.parquet")
    msg = ""
    if os.path.exists(old_path):
        old = pd.read_parquet(old_path, columns=[time_col, group_col, target_col])
        old[time_col] = pd.to_datetime(old[time_col])
        old = old[(old[time_col] >= WINDOW_START) & (old[time_col] <= VAL_END)]
        new = part.loc[part[time_col] <= VAL_END, [time_col, group_col, target_col]]
        m = old.merge(new, on=[time_col, group_col], how="outer", suffixes=("_old", "_new"), indicator=True)
        only_old, only_new = int((m["_merge"] == "left_only").sum()), int((m["_merge"] == "right_only").sum())
        both = m[m["_merge"] == "both"]
        changed = int((both[f"{target_col}_old"] != both[f"{target_col}_new"]).sum())
        tot["only_old"] += only_old; tot["only_new"] += only_new; tot["changed"] += changed; tot["compared"] += len(both)
        msg = f" | vs old: only_old={only_old}, only_new={only_new}, changed={changed}"
        del old, new, m, both
    print(f"  chunk {i+1}/{len(chunks)}: {len(part):,} rows, {part[group_col].nunique():,} series{msg}")
    del part
    gc.collect()

print("\nHistory consistency (dates", WINDOW_START.date(), "->", VAL_END.date(), "):", tot)
if tot["only_old"] or tot["only_new"] or tot["changed"]:
    print("WARNING: the table's older history differs from what the model was trained on. "
          "A small number of changed cells = late corrections; large numbers = the table was rebuilt, investigate before forecasting.")

### Calendar through Dec 31
Same recipe as Step 1. Festival flags cannot be filled forward — the source table must already hold them for Dec 11–31.

In [ ]:
cal_src = df.filter((F.col(time_col) >= str(TRAIN_START.date())) & (F.col(time_col) <= str(FORECAST_END_NEW.date())))
cal = (cal_src.group_by(time_col)
              .agg(*[F.max(F.col(c)).alias(f"C{i}") for i, c in enumerate(festive_covariates)])
              .to_pandas())
cal.columns = [time_col] + festive_covariates
cal[time_col] = pd.to_datetime(cal[time_col])
cal = cal.sort_values(time_col).reset_index(drop=True)

full_idx = pd.date_range(TRAIN_START, FORECAST_END_NEW, freq=FREQ)
missing = full_idx.difference(cal[time_col])
assert len(missing) == 0, (f"Calendar is missing {len(missing)} dates (first: {list(missing[:3])}). "
                           f"Extend MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES to {FORECAST_END_NEW.date()} first.")
assert cal[festive_covariates].notna().all().all(), "Festive columns contain nulls."

dt = cal[time_col].dt
cal["DOW_SIN"] = np.sin(2 * np.pi * dt.dayofweek / 7).astype(np.float32)
cal["DOW_COS"] = np.cos(2 * np.pi * dt.dayofweek / 7).astype(np.float32)
cal["IS_MONTH_END"]      = (dt.day >= dt.days_in_month - 2).astype(np.float32)
cal["IS_MONTH_START"]    = (dt.day <= 3).astype(np.float32)
cal["DAYS_TO_MONTH_END"] = ((dt.days_in_month - dt.day) / 31.0).astype(np.float32)
cal[festive_covariates] = cal[festive_covariates].astype(np.float32)
cal = cal[[time_col] + future_covariates]

# the covariates must be exactly what the model was trained with: same names, same order
assert list(cal.columns[1:]) == list(future_covariates)

# unchanged on the dates the old calendar already covered?
old_cal = pd.read_parquet(OLD_CALENDAR)
old_cal[time_col] = pd.to_datetime(old_cal[time_col])
ov = old_cal.merge(cal, on=time_col, suffixes=("_old", "_new"))
diff_cols = [c for c in future_covariates
             if not np.allclose(ov[f"{c}_old"].to_numpy(np.float64), ov[f"{c}_new"].to_numpy(np.float64), atol=1e-6)]
print("covariates whose values changed on dates the old calendar covered:", diff_cols or "none")

cal.to_parquet(NEW_CALENDAR, index=False)
fc = cal[(cal[time_col] > ORIGIN)]
print(f"Calendar: {cal[time_col].min().date()} -> {cal[time_col].max().date()} ({len(cal)} days), {len(future_covariates)} future covariates")
for anchor in ["N", "D"]:
    print(f"  '{anchor}' falls on:", [d.date().isoformat() for d in fc.loc[fc[anchor] != 0, time_col]])
print("\nFlags active Dec 11 -> Dec 31:",
      {c: int((cal.loc[cal[time_col] > pd.Timestamp('2026-12-10'), c] != 0).sum()) for c in festive_covariates
       if (cal.loc[cal[time_col] > pd.Timestamp('2026-12-10'), c] != 0).any()} or "none")

In [ ]:
session.close()
print("Snowflake session closed.")
print("\nNext: run 04_forecast_new_origin.ipynb (ORIGIN = 2026-10-31).")